# Chapter 41: unsupervised learning

Freshly compute hand PCA/HMM updates, rolling PCA, explained variance versus prediction, one 3,000-day regime path and a post hoc 50-seed diagnostic. The seeds 4200–4249 are not a frozen-summary replay. These multiple fits can be expensive; no fresh runtime measurement was made for this language revision. Save two new figures.

The causal rule signals at close t using data through t, trades at close t+1 and earns the following return, charging 10 bp per one-way traded notional. Rules marked as leaking or impossible deliberately violate that information contract.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch41_unsupervised_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

from quant_course import unsupervised_tools as ut
ch41 = load_chapter('ch41_unsupervised')

## 1. Hand PCA and HMM filtering

Check eigenvalues (3,1), component orientation and the posterior state probabilities for the specified −0.025 observation. Component signs can be chosen consistently without changing their subspace.

In [ ]:
hp, hh = ch41.hand_pca(), ch41.hand_hmm()
display(pd.DataFrame({k: {"eigenvalues": v["eigenvalues"], "PC1": [round(r[0], 4) for r in v["loadings"]],
                          "ratio": v["explained_ratio"]} for k, v in hp.items()}).T)
assert hp["cov_2_1_1_2"]["eigenvalues"] == [3.0, 1.0] and hp["cov_2_1_1_2"]["loadings"][0][1] > 0 > hp["cov_2_1_1_2"]["loadings"][1][1]
print(hh["posterior"])
assert np.allclose(hh["posterior"]["-0.025"], [0.057059, 0.942941], atol=1e-6)

## 2. Training-only versus full-sample PCA

For 50-asset factor/noise panels, compare rolling windows of 60 and 250 observations. Contrast training explained variance with out-of-sample reconstruction and an explicitly leaking full-sample fit. Save the first new figure.

In [ ]:
R, B = ut.simulate_factor_panel(1000, 50, factor_vols=(0.010, 0.005, 0.003), idio_vol=0.015, seed=4101)
noise = pd.DataFrame(np.random.default_rng(4103).normal(0, 0.015, (1000, 50)), index=R.index, columns=R.columns)
pca = {(p, w): ch41.rolling_pca(panel, w) for p, panel in (("factor", R), ("noise", noise)) for w in (60, 250)}
display(pd.DataFrame({f"{p} T={w}": {k: v[k] for k in ("in_sample_explained_mean", "oos_r2_mean", "full_sample_fit_on_test_r2_mean")}
                      for (p, w), v in pca.items()}).T.round(3))
assert round(pca[("noise", 60)]["in_sample_explained_mean"], 3) == 0.186 and round(pca[("noise", 250)]["oos_r2_mean"], 3) == 0.060
labels_ = [f"{p}\nT={w}" for p, w in pca]
xs = np.arange(len(pca))
plt.bar(xs - 0.2, [v["in_sample_explained_mean"] for v in pca.values()], 0.4, color="#7F7F7F", label="in-sample (k=3)")
plt.bar(xs + 0.2, [v["oos_r2_mean"] for v in pca.values()], 0.4, color="#247F82", label="out-of-sample R2")
plt.xticks(xs, labels_); plt.ylabel("share of variance"); plt.legend(); plt.title("CONSTRUCTED 50-asset panels"); plt.savefig(RUN_DIR / "ch41_unsupervised_cell05.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 3. Explained variance is not predictive value

Compare covariance/correlation PCA and the number of components in the stated predictive construction. A high-variance direction need not contain the useful signal.

In [ ]:
vp = ch41.variance_vs_prediction()
display(pd.DataFrame(vp).T)
assert vp["cov_k1"]["oos_r2"] < 0.01 < vp["corr_k1"]["oos_r2"] < vp["cov_k2"]["oos_r2"]

## 4. Regime timing

Rules labeled [leaks], [impossible] or [oracle params] use information unavailable for that trade. Compare lagged filtering with those controls without choosing an implementable winner from a leaked path. Save the second new wealth figure.

In [ ]:
reg = ch41.regime_case()
display(pd.DataFrame(reg["rules"]).T[["ann_return", "sharpe", "max_drawdown", "turnover_ann", "state_hit_rate"]].round(3))
assert round(reg["rules"]["filtered_prob_lagged"]["sharpe"], 2) == -0.51
assert reg["rules"]["smoothed_prob_same_day [leaks]"]["sharpe"] > reg["rules"]["smoothed_prob_lagged [leaks]"]["sharpe"]
idx = reg["_series"]["test_idx"]
for name in ("buy_and_hold", "smoothed_prob_same_day [leaks]", "filtered_prob_lagged", "trailing_vol_lagged"):
    plt.plot(idx, reg["_curves"][name], label=name)
plt.yscale("log"); plt.xlabel("date (test)"); plt.ylabel("wealth after 10 bp (log)"); plt.legend(fontsize=7)
plt.title("CONSTRUCTED regime market, seed 4102"); plt.savefig(RUN_DIR / "ch41_unsupervised_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 5. Fifty-seed post hoc diagnostic

The final call freshly evaluates 50 seeds and displays summaries. Its purpose and post hoc D11 identity remain separate from the initial path; median Sharpe near zero for lagged filtering is retained rather than replaced with the leaking control's result.

In [ ]:
mc = ch41.regime_monte_carlo()
display(pd.DataFrame({k: v for k, v in mc.items() if k != "n_seeds"}).T.round(2))
assert round(mc["smoothed_prob_same_day [leaks]"]["median_sharpe"], 2) == 0.76 and round(mc["filtered_prob_lagged"]["median_sharpe"], 2) == -0.03